<h1>Notebook para Ornithinimicrobiaceae </h1>

In [1]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import hamming
import gudhi as gd
import seaborn as sns
from statistics import mode
import multiprocessing
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
import gudhi as gd
# from scipy.spatial.distance import hamming
import plotly.graph_objs as go
import networkx as nx
import plotly.graph_objects as go
import plotly.io as pio
from networkx.utils import not_implemented_for, pairwise
from concurrent.futures import ThreadPoolExecutor, as_completed
import plotly
import plotly.express as px
from module_functions import *

<h2>Se extrae la información </h2>

In [2]:
info=pd.read_csv('data/pscpOrinth.blast',sep='\t',header=None)
names=pd.read_csv('data/Orinth.ids',sep='\t',dtype='object',header=None)

<h2>Se crean los data frames</h2>

In [3]:
df_by_pathway,df_by_pathway_drop_duplicate,representative_genomes=get_df_by_pathway(info)

In [4]:
print('n genomas:',len(df_by_pathway['3PGA_AMINOACIDS'].index))
cont=0
cont2=0
for i,j in df_by_pathway.items():
    cont2+=len(j.columns)
    cont+=1
    print(i)
print('n rutas',cont)
print('n genes',cont2)

n genomas: 21
Glycolysis
OXALACETATE_AMINOACIDS
PPP
ALPHAKETOGLUTARATE_AMINOACIDS
TCA
﻿Glycolysis
R5P_AMINOACIDS
3PGA_AMINOACIDS
PYR_THR_AA
E4P_AMINO_ACIDS
n rutas 10
n genes 98


<h2> Se crea el complejo simplicial</h2>

In [5]:
complex_pathways=get_complex_by_pathways(df_by_pathway_drop_duplicate)

In [9]:
complex_pathways['R5P_AMINOACIDS']
for i in complex_pathways.keys():
    contador=0
    for j in complex_pathways[i][0]:
        if j[0]==1:
            contador+=1
    print(i,contador)

Glycolysis 1
OXALACETATE_AMINOACIDS 2
PPP 2
ALPHAKETOGLUTARATE_AMINOACIDS 3
TCA 1
﻿Glycolysis 0
R5P_AMINOACIDS 1
3PGA_AMINOACIDS 1
PYR_THR_AA 2
E4P_AMINO_ACIDS 0


In [6]:
#plot_all_bar_code_pathways(complex_pathways)

<h2>Se buscan los uno hoyos</h2>

In [10]:
holes_no_names,holes_with_names,holes_per_filt_lvl=get_holes_by_pathways(complex_pathways,df_by_pathway_drop_duplicate)

se buscan ciclos en el tiempo 2.0 para Glycolysis
se buscan ciclos en el tiempo 3.0 para OXALACETATE_AMINOACIDS
Se encontró el ciclo [9, 6, 1, 0] en el tiempo 2.0 para OXALACETATE_AMINOACIDS
se buscan ciclos en el tiempo 6.0 para OXALACETATE_AMINOACIDS
Se encontró el ciclo [1, 12, 6, 5, 4, 3, 2] en el tiempo 1.0 para Glycolysis
Terminó la ruta  Glycolysis
Se encontró el ciclo [18, 3, 6, 0] en el tiempo 5.0 para OXALACETATE_AMINOACIDS
Terminó la ruta  OXALACETATE_AMINOACIDS
se buscan ciclos en el tiempo 2.0 para PPP
Se encontró el ciclo [15, 7, 9, 6] en el tiempo 1.0 para PPP
Se encontró el ciclo [8, 6, 15, 10, 1, 0] en el tiempo 1.0 para PPP
se buscan ciclos en el tiempo 2.0 para ALPHAKETOGLUTARATE_AMINOACIDS
Se encontró el ciclo [19, 14, 10, 5] en el tiempo 1.0 para ALPHAKETOGLUTARATE_AMINOACIDS
Se encontró el ciclo [19, 8, 6, 5] en el tiempo 1.0 para ALPHAKETOGLUTARATE_AMINOACIDS
se buscan ciclos en el tiempo 3.0 para ALPHAKETOGLUTARATE_AMINOACIDS
Terminó la ruta  PPP
se buscan ciclo

In [31]:
resum=get_resum_df(df_by_pathway,representative_genomes,names,complex_pathways,holes_with_names)
resum

/opt/conda/envs/TDA/lib/python3.7/site-packages/pandas/core/internals/blocks.py:937: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr_value = np.asarray(value)


,num_columnas,num_hoyos,ciclos,ciclos por genero,duración hoyos
Glycolysis,10,1,"[[[Serinicoccus sp. CUA-874 CUA-874, Ornithini...","[[[Serinicoccus, Ornithinimicrobium], Ornithin...",[1.0]
OXALACETATE_AMINOACIDS,16,2,"[[Ornithinimicrobium sp. na, Ornithinimicrobiu...","[[Ornithinimicrobium, Ornithinimicrobium, Orni...",[1.0]
PPP,8,2,"[[Ornithinimicrobium cerasi USBA17B2, [Ornithi...","[[Ornithinimicrobium, [Ornithinimicrobium, Orn...",[1.0]
ALPHAKETOGLUTARATE_AMINOACIDS,11,3,"[[Ornithinimicrobium cerasi USBA17B2, Ornithin...","[[Ornithinimicrobium, Ornithinimicrobium, Orni...",[1.0]
TCA,11,1,"[[Serinicoccus hydrothermalis JLT9, [Serinicoc...","[[Serinicoccus, [Serinicoccus, Ornithinimicrob...",[1.0]
﻿Glycolysis,1,0,[],[],[]
R5P_AMINOACIDS,9,1,"[[Serinicoccus sp. CNJ-927 CNJ-927, Ornithinim...","[[Serinicoccus, Ornithinimicrobium, [Ornithini...",[1.0]
3PGA_AMINOACIDS,5,1,"[[Ornithinimicrobium sp. na, Ornithinimicrobiu...","[[Ornithinimicrobium, Ornithinimicrobium, [Orn...",[1.0]
PYR_THR_AA,10,2,"[[Ornithinimicrobium sp. na, Ornithinimicrobiu...","[[Ornithinimicrobium, Ornithinimicrobium, Seri...",[1.0]
E4P_AMINO_ACIDS,17,0,[],[],[]


In [32]:
generos=set()
for i in traduced_df(df_by_pathway['R5P_AMINOACIDS'],names).index:
    generos.add(i.split(' ')[0])
df_count_genomes=pd.DataFrame(index=generos)
df_count_genomes['conteo']=0
for path in resum.index:
    ciclos_path=resum.loc[path,'ciclos por genero']
    for cic in ciclos_path:
        for vert in cic:
            if type(vert)==str:
                df_count_genomes.loc[vert,'conteo']=df_count_genomes.loc[vert,'conteo']+1
            else:
                for i in vert:
                    df_count_genomes.loc[i,'conteo']=df_count_genomes.loc[i,'conteo']+1
(df_count_genomes.sort_values(by='conteo',ascending=False)/df_count_genomes.sum())*100

,conteo
Ornithinimicrobium,78.409091
Serinicoccus,21.590909


In [33]:
generos=set()
for i in traduced_df(df_by_pathway['R5P_AMINOACIDS'],names).index:
    generos.add(i.split(' ')[0])
df_count_genomes=pd.DataFrame(index=traduced_df(df_by_pathway['R5P_AMINOACIDS'],names).index)
df_count_genomes['conteo']=0
for path in resum.index:
    ciclos_path=resum.loc[path,'ciclos']
    for cic in ciclos_path:
        for vert in cic:
            if type(vert)==str:
                df_count_genomes.loc[vert,'conteo']=df_count_genomes.loc[vert,'conteo']+1
            else:
                for i in vert:
                    df_count_genomes.loc[i,'conteo']=df_count_genomes.loc[i,'conteo']+1
df_count_genomes.sort_values(by='conteo',ascending=False)

,conteo
R5P_AMINOACIDS,
Ornithinimicrobium sp. na,11
Ornithinimicrobium sp. na,11
Ornithinimicrobium sp. na,11
Ornithinimicrobium sp. na,11
Ornithinimicrobium cerasi USBA17B2,9
Serinicoccus chungangensis CD08_5,8
Ornithinimicrobium humiphilum DSM 12362,7
Serinicoccus hydrothermalis JLT9,6
Ornithinimicrobium avium AMA3305,6


In [35]:
long=[]
for k in resum.index:
    for i in resum.loc[k,'ciclos']:
        long.append(len(i))
print(set(long))
long.count(6)

{4, 6, 7}


1

<h2>Se ven los elementos que hacen el hoyo</h2>

In [28]:
df_pathway_make_hole=get_df_by_pathway_make_hole_droped_cols(df_by_pathway_drop_duplicate,holes_with_names)
df_count_genes_participation=pd.DataFrame()
for path in df_pathway_make_hole.keys():
    print('los genes que participan en los 1-hoyos para',path,'son:')
    genes=[]
    for df in df_pathway_make_hole[path]:
        genes.extend(list(df.columns))
    set_count_genes=set()
    for gen in genes:
        df_count_genes_participation.loc[gen,'conteo']=genes.count(gen)
        df_count_genes_participation.loc[gen,'ruta']=path
        set_count_genes.add(gen+':'+str(genes.count(gen)))
    print(str(set_count_genes))

los genes que participan en los 1-hoyos para Glycolysis son:
{'Phosphofructokinase:1', 'phosphoglycerate_mutase:1', 'Glucose_kinase:1'}
los genes que participan en los 1-hoyos para OXALACETATE_AMINOACIDS son:
{'succinyldiaminopimelate_transaminase:2', 'Methionine_synthase:1', 'Aspartate_transaminase:2', 'Asparagine_synthase:1', 'cystathione_beta_lyase:1', 'Homoserine_O_succinyltransferase:2', 'argininosuccinate_lyase:1', 'cystathione_gamma_synthetase:1'}
los genes que participan en los 1-hoyos para PPP son:
{'Ribose_5_phosphate_isomerase:1', '6_phosphogluconolactonase:1', 'phosphogluconate_dehydrogenase:2', 'Glucose_6_phosphate_dehydrogenase:1'}
los genes que participan en los 1-hoyos para ALPHAKETOGLUTARATE_AMINOACIDS son:
{'glutamate:1', 'Acetylornithine_aminotransferase:3', 'glutamate_synthase:3'}
los genes que participan en los 1-hoyos para TCA son:
{'Citrate_synthase:1', 'Aconitate_hydratase:1', 'fumarate_hydratase:1'}
los genes que participan en los 1-hoyos para ﻿Glycolysis son:


In [30]:
total=df_count_genes_participation.sum()['conteo']

df_count_genes_participation.to_csv('1.csv')

In [27]:
df_by_pathway['Glycolysis']

,Fructose_biphosphate_aldolase,Pyruvate_kinase,glyceraldehyde3phosphate_dehydrogenase,Glucose_6_phosphate_isomerase,enolase,phosphoglycerate_kinase,phosphoglycerate_mutase,Triosephosphate_isomerase,Glucose_kinase,Phosphofructokinase
Glycolysis,,,,,,,,,,
666666.100004,1.0,1.0,1.0,0.0,1.0,1.0,2,1.0,4,1.0
666666.100005,1.0,1.0,1.0,0.0,1.0,1.0,2,1.0,4,1.0
666666.100006,1.0,1.0,1.0,0.0,1.0,1.0,1,1.0,3,1.0
666666.100007,1.0,1.0,1.0,0.0,1.0,1.0,2,1.0,3,1.0
666666.100008,1.0,1.0,1.0,0.0,1.0,1.0,2,1.0,2,1.0
666666.100010,1.0,1.0,1.0,0.0,1.0,1.0,2,1.0,2,2.0
666666.100019,1.0,1.0,1.0,0.0,1.0,1.0,2,1.0,5,2.0
666666.100020,1.0,1.0,1.0,0.0,1.0,1.0,1,1.0,5,2.0
666666.100023,1.0,1.0,1.0,0.0,1.0,1.0,2,1.0,6,3.0


In [16]:
traduced_df(df_pathway_make_hole['R5P_AMINOACIDS'][0],names)


,histidinol_dehydrogenase,phosphoribosyl_AMP_cyclohydrolase,histidinol_phosphatase,ATP_phosphoribosyltransferase,histidinolphosphate_aminotransferase,imidazoleglycerol_phosphate_dehydratase,imidazole_glycerol_phosphate_synthase_H,ribosephosphate_pyrophosphokinase,phosphoribosyl_isomerase_A
R5P_AMINOACIDS,,,,,,,,,
Serinicoccus sp. CNJ-927 CNJ-927,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,2
Ornithinimicrobium sp. na,1.0,0.0,1.0,0.0,1.0,1.0,2.0,0.0,2
Ornithinimicrobium humiphilum DSM 12362,1.0,0.0,1.0,1.0,2.0,1.0,2.0,1.0,2
Serinicoccus chungangensis CD08_5,1.0,1.0,1.0,1.0,2.0,1.0,2.0,1.0,2


<h2>Se buscan uno hoyos pero al separar por genero y ruta metabólica</h2>

In [17]:
df_by_genus_pathway,df_by_genus_pathway_drop_duplicate,representative_genomes_genus_pathway=get_df_by_genus_pathway(df_by_pathway,names)
complex_genus_pathways=get_complex_by_pathways(df_by_genus_pathway_drop_duplicate)

In [18]:
traduced_df(df_by_genus_pathway['Serinicoccus_ALPHAKETOGLUTARATE_AMINOACIDS'],names)

,Glutamine_2-oxoglutarate,Glutamate_5_kinase,Glutamate-5-semialdehyde_dehydrogenase,Acetylornithine_aminotransferase,glutamate_synthase,glutamate,Argininosuccinate,argininosuccinate_lyase,Pyrroline-5-carboxylate_reductase,Glutamine_synthetase,Ornithine_carbamoyltransferase
Serinicoccus_ALPHAKETOGLUTARATE_AMINOACIDS,,,,,,,,,,,
Serinicoccus chungangensis CD08_5,1.0,1.0,1.0,6.0,1.0,0.0,1.0,1.0,1.0,3,1.0
Serinicoccus hydrothermalis JLT9,1.0,1.0,1.0,8.0,2.0,0.0,1.0,1.0,1.0,3,1.0
Serinicoccus sp. CUA-874 CUA-874,1.0,1.0,1.0,5.0,1.0,0.0,1.0,1.0,1.0,4,1.0
Serinicoccus sp. CNJ-927 CNJ-927,1.0,0.0,1.0,7.0,1.0,0.0,1.0,1.0,1.0,4,1.0


In [19]:
holes_genus_no_names,holes_genus_with_names,holes_genus_per_filt_lvl=get_holes_by_pathways(complex_genus_pathways,df_by_genus_pathway_drop_duplicate)

se buscan ciclos en el tiempo 6.0 para Ornithinimicrobium_OXALACETATE_AMINOACIDS
se buscan ciclos en el tiempo 2.0 para Ornithinimicrobium_ALPHAKETOGLUTARATE_AMINOACIDS
se buscan ciclos en el tiempo 2.0 para Ornithinimicrobium_PPP
Se encontró el ciclo [11, 3, 5, 2] en el tiempo 1.0 para Ornithinimicrobium_PPP
Terminó la ruta  Ornithinimicrobium_PPP
Se encontró el ciclo [15, 10, 6, 1] en el tiempo 1.0 para Ornithinimicrobium_ALPHAKETOGLUTARATE_AMINOACIDS
Se encontró el ciclo [15, 4, 2, 1] en el tiempo 1.0 para Ornithinimicrobium_ALPHAKETOGLUTARATE_AMINOACIDS
Terminó la ruta  Ornithinimicrobium_ALPHAKETOGLUTARATE_AMINOACIDS
se buscan ciclos en el tiempo 3.0 para Ornithinimicrobium_3PGA_AMINOACIDS
Se encontró el ciclo [10, 1, 5, 2] en el tiempo 2.0 para Ornithinimicrobium_3PGA_AMINOACIDS
Terminó la ruta  Ornithinimicrobium_3PGA_AMINOACIDS
Se encontró el ciclo [0, 14, 16, 2] en el tiempo 5.0 para Ornithinimicrobium_OXALACETATE_AMINOACIDS
Se encontró el ciclo [4, 7, 5, 2] en el tiempo 5.0 p

In [20]:
resum_genus=get_resum_df(df_by_genus_pathway,representative_genomes_genus_pathway,names,complex_genus_pathways,holes_genus_with_names)
resum_genus

/opt/conda/envs/TDA/lib/python3.7/site-packages/pandas/core/internals/blocks.py:937: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr_value = np.asarray(value)


,num_columnas,num_hoyos,ciclos,ciclos por genero,duración hoyos
Serinicoccus_OXALACETATE_AMINOACIDS,16,0,[],[],[]
Ornithinimicrobium_OXALACETATE_AMINOACIDS,16,2,"[[Ornithinimicrobium pratense W204, Ornithinim...","[[Ornithinimicrobium, Ornithinimicrobium, Orni...",[1.0]
Serinicoccus_E4P_AMINO_ACIDS,17,0,[],[],[]
Ornithinimicrobium_E4P_AMINO_ACIDS,17,0,[],[],[]
Serinicoccus_ALPHAKETOGLUTARATE_AMINOACIDS,11,0,[],[],[]
Ornithinimicrobium_ALPHAKETOGLUTARATE_AMINOACIDS,11,2,"[[Ornithinimicrobium cerasi USBA17B2, Ornithin...","[[Ornithinimicrobium, Ornithinimicrobium, Orni...",[1.0]
Serinicoccus_﻿Glycolysis,1,0,[],[],[]
Ornithinimicrobium_﻿Glycolysis,1,0,[],[],[]
Serinicoccus_PYR_THR_AA,10,0,[],[],[]
Ornithinimicrobium_PYR_THR_AA,10,0,[],[],[]


<h1>Para obtener la información de los 1-hoyos de la ruta R5P AMINOACIDS:</h1>

In [22]:
R5P_df_by_pathway={'R5P_AMINOACIDS':df_by_pathway['R5P_AMINOACIDS']}
R5P_df_by_pathway_drop_duplicate={'R5P_AMINOACIDS':df_by_pathway_drop_duplicate['R5P_AMINOACIDS']}
R5P_representative_genomes={'R5P_AMINOACIDS':representative_genomes['R5P_AMINOACIDS']}
R5P_complex_pathways={'R5P_AMINOACIDS':complex_pathways['R5P_AMINOACIDS']}

In [23]:
R5P_holes_no_names,R5P_holes_with_names,R5P_holes_per_filt_lvl=get_holes_by_pathways(R5P_complex_pathways,R5P_df_by_pathway_drop_duplicate)

se buscan ciclos en el tiempo 4.0 para R5P_AMINOACIDS
Se encontró el ciclo [3, 7, 4, 0] en el tiempo 3.0 para R5P_AMINOACIDS
Terminó la ruta  R5P_AMINOACIDS


In [24]:
get_resum_df(R5P_df_by_pathway,R5P_representative_genomes,names,R5P_complex_pathways,R5P_holes_with_names)

/opt/conda/envs/TDA/lib/python3.7/site-packages/pandas/core/internals/blocks.py:937: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr_value = np.asarray(value)


,num_columnas,num_hoyos,ciclos,ciclos por genero,duración hoyos
R5P_AMINOACIDS,9,1,"[[Serinicoccus sp. CNJ-927 CNJ-927, Ornithinim...","[[Serinicoccus, Ornithinimicrobium, [Ornithini...",[1.0]


<h3>Género por R5P</h3>

In [25]:
R5P_df_by_genus_pathway,R5P_df_by_genus_pathway_drop_duplicate,R5P_representative_genomes_genus_pathway=get_df_by_genus_pathway(R5P_df_by_pathway,names)
R5P_complex_genus_pathways=get_complex_by_pathways(R5P_df_by_genus_pathway_drop_duplicate)

In [26]:
R5P_holes_genus_no_names,R5P_holes_genus_with_names,R5P_holes_genus_per_filt_lvl=get_holes_by_pathways(R5P_complex_genus_pathways,R5P_df_by_genus_pathway_drop_duplicate)

In [28]:
get_resum_df(R5P_df_by_genus_pathway,R5P_representative_genomes_genus_pathway,names,R5P_complex_genus_pathways,R5P_holes_genus_per_filt_lvl)

,num_columnas,num_hoyos,ciclos,ciclos por genero,duración hoyos
Serinicoccus_R5P_AMINOACIDS,9,0,[],[],[]
Ornithinimicrobium_R5P_AMINOACIDS,9,0,[],[],[]
